# AutoAssess — Train YOLOv8-seg on VehiDE (Colab)

Trains the damage-detection model on the **VehiDE** dataset (7 structural/cosmetic damage classes — scratch, dent, torn, broken_lamp, missing_part, punctured, shattered_glass) instead of CarDD's 6, to cover damage types CarDD has no equivalent for.

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

**You'll need a Kaggle API token.** Get one at kaggle.com -> Settings -> API -> Create New Token (downloads `kaggle.json`). Add it as a Colab secret (key icon in the left sidebar) named `KAGGLE_JSON` containing the full file contents, or upload `kaggle.json` when prompted in the cell below.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/DhyeyTandel/autoassess.git
%cd autoassess
!pip install -q -e .
!pip install -q kaggle

## 2. Kaggle credentials

Tries a Colab secret named `KAGGLE_JSON` first; falls back to a manual file upload if the secret isn't set.

In [ ]:
import json
import os
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"

wrote_from_secret = False
try:
    from google.colab import userdata
    raw = userdata.get("KAGGLE_JSON")
    json.loads(raw)  # validate it's real JSON before writing
    kaggle_json_path.write_text(raw)
    wrote_from_secret = True
    print("Wrote ~/.kaggle/kaggle.json from Colab secret KAGGLE_JSON.")
except Exception as e:
    print(f"Could not read Colab secret KAGGLE_JSON ({e!r}). Falling back to file upload.")

if not wrote_from_secret:
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    fname = next(iter(uploaded))
    kaggle_json_path.write_bytes(uploaded[fname])

os.chmod(kaggle_json_path, 0o600)
print("Kaggle credentials ready at", kaggle_json_path)

## 3. Download + convert VehiDE

Same scripts used locally — `scripts/download_vehide.py` (Kaggle API) and `autoassess-convert-vehide` (VIA JSON -> YOLO-seg format).

In [ ]:
!python scripts/download_vehide.py --config configs/base.yaml

In [ ]:
!autoassess-convert-vehide --config configs/base.yaml --dataset-config configs/vehide.yaml

## 4. Train YOLOv8-seg on VehiDE

Uses the repo's own defaults (`yolov8s-seg`, 50 epochs, batch 8, imgsz 640) — same training script (`train_yolo.py`) as the CarDD/parts models, just pointed at `configs/vehide.yaml`. `device 0` is Colab's GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!autoassess-train-yolo \
  --dataset-config configs/vehide.yaml \
  --model yolov8s-seg.pt \
  --epochs 50 \
  --batch 16 \
  --imgsz 640 \
  --device 0 \
  --name vehide_seg_v1

## 5. Check results

In [ ]:
import json

metrics = json.loads(Path("runs/vehide_seg_v1/metrics.json").read_text())
print(json.dumps({k: v for k, v in metrics.items() if k != "epoch_wall_times_seconds"}, indent=2))

## 6. Package weights + metrics for download

Zips `runs/vehide_seg_v1/weights/{best,last}.pt` and `metrics.json` — download this and extract into `runs/vehide_seg_v1/` in your local repo so eval scripts and `app.py` can pick it up.

In [ ]:
import shutil

shutil.make_archive("vehide_seg_v1_results", "zip", "runs/vehide_seg_v1")
print("Packaged: vehide_seg_v1_results.zip")

from google.colab import files
files.download("vehide_seg_v1_results.zip")